In [ ]:
import numpy as np
import random
import heapq
import matplotlib.pyplot as plt

def generate_labyrinth(width, height):
    """
    Genererar en smalare labyrint med Depth-First Search.
    """
    labyrinth = np.ones((height, width), dtype=int)

    def carve(x, y):
        directions = [(0, 2), (0, -2), (2, 0), (-2, 0)]
        random.shuffle(directions)
        for dx, dy in directions:
            nx, ny = x + dx, y + dy
            if 1 <= nx < height - 1 and 1 <= ny < width - 1 and labyrinth[nx, ny] == 1:
                labyrinth[x + dx // 2, y + dy // 2] = 0  # Carve path
                labyrinth[nx, ny] = 0  # Mark as visited
                carve(nx, ny)

    # Start from (1,1)
    labyrinth[1, 1] = 0
    carve(1, 1)
    return labyrinth

# 1. Generera labyrinten
maze = generate_labyrinth(21, 21)  # Skapa en smalare labyrint med 21x21 dimensioner
print("Genererad Labyrint:")
print(maze)

plt.imshow(maze, cmap='gray_r')
plt.title("Genererad Labyrint")
plt.axis('off')
plt.show()

start_point = (1, 1)
goal_point = (19, 19)

# 2. Markera start och mål i labyrinten
maze_with_markers = maze.copy()
maze_with_markers[start_point] = 2  # Start markeras med 2
maze_with_markers[goal_point] = 3   # Mål markeras med 3

plt.imshow(maze_with_markers, cmap='gray_r')
plt.title("Labyrint med Start (2) och Mål (3)")
plt.axis('off')
plt.show()

def heuristic(a, b):
    """
    Beräknar Euclidean-avståndet mellan två punkter.
    """
    return np.sqrt((b[0] - a[0]) ** 2 + (b[1] - a[1]) ** 2)

def a_star_search(maze, start, goal):
    """
    A*-algoritm för att hitta kortaste vägen i en labyrint.
    """
    rows, cols = maze.shape
    open_list = []
    heapq.heappush(open_list, (0, start))
    came_from = {}
    cost_so_far = {start: 0}

    while open_list:
        _, current = heapq.heappop(open_list)

        if current == goal:
            break

        for dx, dy in [(-1, 0), (1, 0), (0, -1), (0, 1)]:
            neighbor = (current[0] + dx, current[1] + dy)

            if 0 <= neighbor[0] < rows and 0 <= neighbor[1] < cols and maze[neighbor[0]][neighbor[1]] == 0:
                new_cost = cost_so_far[current] + 1
                if neighbor not in cost_so_far or new_cost < cost_so_far[neighbor]:
                    cost_so_far[neighbor] = new_cost
                    priority = new_cost + heuristic(neighbor, goal)
                    heapq.heappush(open_list, (priority, neighbor))
                    came_from[neighbor] = current

    return came_from

def rebuild_path(came_from, start, goal):
    """
    Återuppbygger sökvägen från mål till start.
    """
    path = []
    current = goal
    while current != start:
        path.append(current)
        current = came_from[current]
    path.append(start)
    path.reverse()
    return path

# 3. Kör A*-algoritmen
came_from_test = a_star_search(maze, start_point, goal_point)
print("Kör A*-algoritm...")

# 4. Återuppbygg sökvägen
solved_path = rebuild_path(came_from_test, start_point, goal_point)

# Markera sökvägen i labyrinten
for x, y in solved_path:
    maze_with_markers[x, y] = 4  # Sökvägen markeras med 4

# Visa den lösta labyrinten med sökvägen
plt.imshow(maze_with_markers, cmap='gray_r')
plt.title("Löst Labyrint med A*-algoritm")
plt.axis('off')
plt.show()